# 15 — Structured Streaming: Conceitos

**EP · Data Science Knowledge Base | Big Data e Processamento Distribuído | Notebook 15/16**

**Execução:** use o kernel Python 3, reinicie o kernel e execute as células em ordem. Laboratórios Spark são opcionais e desativados por padrão.

**Abordagem:** os exemplos locais verificam conceitos e não representam um cluster distribuído.


## Materiais complementares

- [Guia Técnico](./reference_materials/15_spark_streaming_conceitos.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_spark_streaming_conceitos.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
Identificar limites do streaming e simular janelas temporais.

Ao final, você deve conseguir interpretar os resultados, justificar as escolhas e explicar suas limitações.

## Fundamentação teórica
Micro-batches, watermark, event-time, checkpoint, idempotência e exatamente-uma-vez condicionado ao sink.

**Atenção metodológica:** execução local facilita entendimento e depuração, mas não comprova ganhos de paralelismo em infraestrutura distribuída. Escalabilidade exige medir dados, nós, rede, memória, partições e operações de shuffle.

In [1]:
from pathlib import Path
import sys
DATASETS = Path('datasets')
assert DATASETS.is_dir(), 'Execute a partir da raiz da área 10 (pasta datasets não encontrada)'
print('Python:', sys.version.split()[0])
print('Pasta dos dados:', DATASETS.as_posix())

Python: 3.14.4
Pasta dos dados: datasets


## Demonstração 1 — construção e observação

In [2]:
import pandas as pd
fluxo=pd.DataFrame({'ts':pd.to_datetime(['2026-01-01 12:00:01','2026-01-01 12:00:07','2026-01-01 12:01:03']),
'evento':['A','A','B']})
fluxo['janela']=fluxo['ts'].dt.floor('1min')
print(fluxo.groupby(['janela','evento']).size())

janela               evento
2026-01-01 12:00:00  A         2
2026-01-01 12:01:00  B         1
dtype: int64


## Interpretação
Observe o esquema, as unidades e os resultados. Antes de concluir que um algoritmo é eficiente, distinga **complexidade computacional**, **custo de comunicação** e **volume de dados**. Os cálculos locais ilustram a semântica; o Spark, quando disponível, ilustra a API distribuída.

## Demonstração 2 — validação e limitações

In [3]:
print('Tempo do evento difere de tempo de processamento.')
print('Watermark regula dados atrasados; checkpoint ajuda recuperação; garantias dependem da origem e do destino.')
print('Não confundir demonstração pandas com execução contínua distribuída.')

Tempo do evento difere de tempo de processamento.
Watermark regula dados atrasados; checkpoint ajuda recuperação; garantias dependem da origem e do destino.
Não confundir demonstração pandas com execução contínua distribuída.


## Boas práticas e limitações

- Execute com versões registradas, datasets identificados e caminhos relativos.
- Diferencie execução local, emulação lógica e infraestrutura distribuída real.
- Documente nulos, chaves, cardinalidade, custos e limitações das métricas.
- Não publique datasets ou resultados cuja redistribuição não esteja autorizada.
- Evite `collect()` de grandes tabelas e não confunda uma execução `local[*]` com cluster multinó.


## Exercícios
1. Explique que parte do exemplo poderia ser distribuída por chave ou partição e qual é o custo do shuffle.
2. Descreva uma validação de qualidade para os dados usados.
3. Indique um cenário em que o exemplo local deixaria de ser adequado.

## Respostas comentadas (orientativas)
1. Agregações por chave podem combinar resultados parciais, mas exigem redistribuir dados com a mesma chave; filtros e projeções usualmente evitam shuffle.
2. Verifique contagem, unicidade de chaves, percentual de valores ausentes, tipos, intervalos e consistência das unidades.
3. Quando dados, latência, concorrência ou exigências de falha excederem a capacidade de uma máquina; nesse caso, considere Spark/Hadoop e meça os custos adicionais.

## Síntese

O capítulo 15 reuniu conceitos e verificações de **tempo de evento e watermark**. Os exemplos de pequena escala não comprovam desempenho em cluster. Consulte apenas referências complementares efetivamente publicadas e documentadas.


## Revisão científica — auditoria e aplicação

**Ponto de atenção deste capítulo:** watermarks. Esta seção complementa os exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **watermarks** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — Semântica event-time e atraso de eventos

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.

In [4]:
# Simulação DIDÁTICA de chegada fora de ordem, não uma implementação do watermark Spark.
from datetime import datetime, timedelta
inicio = datetime(2026,1,1,12,0)
atraso_admitido = timedelta(minutes=5)
eventos_chegando = [inicio + timedelta(minutes=m) for m in [0,3,12,2,13]]
maior_event_time = None
fora_do_limite = []
for indice, instante_evento in enumerate(eventos_chegando, start=1):
    limite_anterior = (maior_event_time - atraso_admitido) if maior_event_time is not None else None
    candidato_tardio = limite_anterior is not None and instante_evento < limite_anterior
    fora_do_limite.append(candidato_tardio)
    maior_event_time = max(maior_event_time, instante_evento) if maior_event_time else instante_evento
    print('Chegada',indice,'| tempo do evento:',instante_evento.time(),
          '| limite anterior:',limite_anterior.time() if limite_anterior else 'indefinido',
          '| candidato tardio:',candidato_tardio)
assert fora_do_limite==[False,False,False,True,False]
print('Em Spark, a efetiva remoção de estado/dados depende da agregação e do modo de saída.')


Chegada 1 | tempo do evento: 12:00:00 | limite anterior: indefinido | candidato tardio: False
Chegada 2 | tempo do evento: 12:03:00 | limite anterior: 11:55:00 | candidato tardio: False
Chegada 3 | tempo do evento: 12:12:00 | limite anterior: 11:58:00 | candidato tardio: False
Chegada 4 | tempo do evento: 12:02:00 | limite anterior: 12:07:00 | candidato tardio: True
Chegada 5 | tempo do evento: 12:13:00 | limite anterior: 12:07:00 | candidato tardio: False
Em Spark, a efetiva remoção de estado/dados depende da agregação e do modo de saída.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **watermarks** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Laboratório opcional — Spark local real

Requer `RUN_SPARK_LABS=1` e `SPARK_LAB_PYTHON` apontando para um ambiente isolado com PySpark e Java. O teste falha explicitamente se executado incorretamente; resultados locais não comprovam cluster distribuído.


In [5]:
# Laboratório Spark real opcional, executado em interpretador isolado.
# Ative SOMENTE quando o ambiente PySpark e o Java já estiverem preparados.
import os, sys, subprocess, textwrap
if os.environ.get('RUN_SPARK_LABS') == '1':
    interpretador = os.environ.get('SPARK_LAB_PYTHON', sys.executable)
    assert os.path.isfile(interpretador), f'Interpretador não encontrado: {interpretador}'
    programa = textwrap.dedent(r'''from pyspark.sql import SparkSession,functions as F
from pyspark.sql.types import StructType,StructField,TimestampType,StringType
from pathlib import Path
import tempfile,json
s=(SparkSession.builder.master('local[2]').appName('ep-area10-15')
   .config('spark.ui.enabled','false').config('spark.sql.shuffle.partitions','2').getOrCreate())
try:
    s.sparkContext.setLogLevel('ERROR')
    esquema=StructType([StructField('ts',TimestampType()),StructField('evento',StringType())])
    with tempfile.TemporaryDirectory(prefix='ep10_stream_') as tmp:
        p=Path(tmp)
        (p/'eventos.json').write_text('\n'.join(json.dumps(x) for x in [
          {'ts':'2026-01-01T12:00:01','evento':'A'},
          {'ts':'2026-01-01T12:00:07','evento':'A'},
          {'ts':'2026-01-01T12:01:03','evento':'B'}]),encoding='utf8')
        fonte=s.readStream.schema(esquema).json(str(p))
        agregado=(fonte.withWatermark('ts','1 minute')
          .groupBy(F.window('ts','1 minute'),F.col('evento')).count())
        nome='ep10_stream_15'
        consulta=(agregado.writeStream.format('memory').queryName(nome)
                  .outputMode('complete').trigger(availableNow=True).start())
        assert consulta.awaitTermination(90), 'Microbatch não terminou no tempo previsto'
        linhas=s.sql(f'SELECT evento,count FROM {nome}').collect()
        valores={r['evento']:r['count'] for r in linhas}
        assert valores=={'A':2,'B':1},valores
        print('Structured Streaming microbatch de arquivo:',valores)
        print('Não houve teste de descarte de eventos atrasados em múltiplos microbatches.')
        print('SPARK_LAB_OK_15')
finally:s.stop()''')
    ambiente = os.environ.copy()
    ambiente.setdefault('SPARK_LOCAL_IP', '127.0.0.1')
    ambiente['PYSPARK_PYTHON'] = interpretador
    ambiente['PYSPARK_DRIVER_PYTHON'] = interpretador
    resultado = subprocess.run([interpretador, '-c', programa], cwd=os.getcwd(),
                               env=ambiente, text=True, capture_output=True, timeout=150)
    if resultado.stdout: print(resultado.stdout.strip())
    if resultado.returncode:
        print(resultado.stderr[-3500:])
        raise RuntimeError('O LABORATÓRIO SPARK 15 FALHOU; não registrar como aprovado.')
    assert 'SPARK_LAB_OK_15' in resultado.stdout, 'Marcador de sucesso Spark não encontrado'
else:
    print('SPARK NÃO EXECUTADO (opcional). Ative RUN_SPARK_LABS=1 e SPARK_LAB_PYTHON.')


Structured Streaming microbatch de arquivo: {'A': 2, 'B': 1}
Não houve teste de descarte de eventos atrasados em múltiplos microbatches.
SPARK_LAB_OK_15
